# Distributed LLM Worker Node — Multi-Worker GPU Simulation
**CSE362 Distributed Computing Project**

This notebook runs on a **single Colab GPU** and simulates **N independent virtual workers**.
Each worker is a Ray actor with its own:
- Ollama LLM connection (shared Ollama server, concurrent requests)
- FAISS-based RAG retriever
- Metrics tracker (active/completed/failed requests)

Each virtual worker registers **independently** with the master on AWS EC2 and sends its own heartbeats.

**Architecture on this node:**
```
    Master (AWS EC2)
         │
         | HTTP via cloudflare tunnels
         ▼  
 ┌────────────────────────────────────┐
 │          Colab GPU Node            │
 │  ┌──────────┐  ┌──────────┐  ...   │
 │  │ Worker-1 │  │ Worker-2 │        │
 │  │ Ray Actor│  │ Ray Actor│        │
 │  │ FastAPI  │  │ FastAPI  │        │
 │  │ RAG+LLM  │  │ RAG+LLM  │        │
 │  └──────────┘  └──────────┘        │
 │        └────────────┘              │
 │           Ollama :11434            │
 └────────────────────────────────────┘
```

## Cell 1 — Configuration
Set your values here. Everything else runs automatically.

In [1]:
# =========================================================
# CONFIGURATION — edit these values
# =========================================================

# AWS EC2 public IP where master.py is running
AWS_SERVER_IP    = "13.63.238.244"
MASTER_API_URL   = f"http://{AWS_SERVER_IP}"

# Ollama model to use (must fit in GPU VRAM)
# Recommended small models: phi3:mini | tinyllama | gemma:2b | qwen:0.5b
MODEL_NAME = "qwen2:0.5b"

# ── Multi-worker simulation ────────────────────────────────
# How many virtual workers to spawn on this GPU.
# Each gets its own Ray actor, FastAPI server, ngrok tunnel,
# and registers independently with the master.
# Recommended: 2-4 (Colab T4 has ~15GB VRAM; phi3:mini uses ~3.8GB)
NUM_VIRTUAL_WORKERS = 3

# Base name — workers will be named gpu-worker-1, gpu-worker-2, …
WORKER_BASE_NAME = "gpu-worker"

# Base port — workers listen on 8001, 8002, 8003, …
BASE_PORT = 8001

# Heartbeat interval in seconds
HEARTBEAT_SECONDS = 8

# GPU fraction each Ray actor requests (keep sum ≤ 1.0)
# With 3 workers: 0.3 each = 0.9 total — leaves headroom for Ollama
GPU_FRACTION_PER_WORKER = round(0.9 / NUM_VIRTUAL_WORKERS, 2)

print("=" * 50)
print(f"Master URL         : {MASTER_API_URL}")
print(f"Model              : {MODEL_NAME}")
print(f"Virtual workers    : {NUM_VIRTUAL_WORKERS}")
print(f"GPU fraction each  : {GPU_FRACTION_PER_WORKER}")
print(f"Ports              : {BASE_PORT} → {BASE_PORT + NUM_VIRTUAL_WORKERS - 1}")
print("=" * 50)

Master URL         : http://13.63.238.244
Model              : qwen2:0.5b
Virtual workers    : 3
GPU fraction each  : 0.3
Ports              : 8001 → 8003


## Cell 2 — Install Dependencies & Ollama

In [2]:
# Install system deps
!sudo apt-get install -y -q zstd curl

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install Python packages
!pip install -q \
    "ray[default]" \
    fastapi \
    "uvicorn[standard]" \
    sentence-transformers \
    faiss-cpu \
    requests \
    nest_asyncio \
    httpx \
    python-multipart



import subprocess, os

# Download cloudflared binary
result = subprocess.run([
    "wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"
], capture_output=True)

# Make executable
os.chmod("/usr/local/bin/cloudflared", 0o755)

# Verify
r = subprocess.run(["cloudflared", "--version"], capture_output=True, text=True)
print(r.stdout)

print("\n✅ All dependencies installed")

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libcurl4 libcurl4-openssl-dev
Suggested packages:
  libcurl4-doc libidn11-dev libldap2-dev librtmp-dev
The following NEW packages will be installed:
  zstd
The following packages will be upgraded:
  curl libcurl4 libcurl4-openssl-dev
3 upgraded, 1 newly installed, 0 to remove and 97 not upgraded.
Need to get 1,474 kB of archives.
After this operation, 1,699 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libcurl4-openssl-dev amd64 7.81.0-1ubuntu1.24 [387 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 curl amd64 7.81.0-1ubuntu1.24 [194 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libcurl4 amd64 7.81.0-1ubuntu1.24 [291 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 1,474 kB in 0s (4,871 kB/s)
d

## Cell 3 — Start Ollama Server & Pull Model

In [3]:
import subprocess
import time
import requests as req

# Start Ollama server in background
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for it to be ready
print("Waiting for Ollama to start...", end="")
for _ in range(30):
    try:
        resp = req.get("http://localhost:11434", timeout=2)
        if resp.status_code == 200:
            print(" ✅ Ollama is up!")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)

# Pull the model
print(f"\nPulling model: {MODEL_NAME} (this may take a few minutes)...")
result = subprocess.run(
    ["ollama", "pull", MODEL_NAME],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
print(f"✅ Model {MODEL_NAME} ready")

# Quick smoke test
test_resp = req.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_NAME, "prompt": "Say hi in 3 words.", "stream": False}
)
print("Smoke test →", test_resp.json()["response"].strip())

Waiting for Ollama to start.... ✅ Ollama is up!

Pulling model: qwen2:0.5b (this may take a few minutes)...

✅ Model qwen2:0.5b ready
Smoke test → Warm Welcome, Hello, How Are You?


## Cell 4 — Initialise Ray (local, no remote cluster needed)

In [4]:
import ray

# Initialise Ray locally on this Colab instance.
# We do NOT connect to a remote Ray cluster; each Colab notebook
# is its own Ray head.  The master on EC2 is contacted via HTTP, not Ray.
if ray.is_initialized():
    ray.shutdown()

ray.init(
    num_cpus=4,
    num_gpus=1,
    ignore_reinit_error=True,
    log_to_driver=False,
)

print("✅ Ray initialised")
print(f"   Available resources: {ray.available_resources()}")

2026-05-11 19:06:18,576	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


✅ Ray initialised
   Available resources: {'node:__internal_head__': 1.0, 'CPU': 4.0, 'object_store_memory': 3977259417.0, 'node:172.28.0.12': 1.0, 'memory': 9280271975.0, 'GPU': 1.0, 'accelerator_type:T4': 1.0}


/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## Cell 5 — Knowledge Base (shared across all workers)
Add as many documents as you like.  Workers share the same corpus but each
builds its own FAISS index so queries are truly independent.

In [5]:
# ── Knowledge base documents ──────────────────────────────────────────────────
# Add domain-specific docs here.  Each string becomes a retrievable chunk.

KNOWLEDGE_BASE = [
    """Distributed systems use multiple nodes to achieve scalability, fault
    tolerance, and high availability.  They communicate over a network and
    must handle partial failures gracefully.""",

    """Ray is an open-source distributed execution framework originally
    designed for reinforcement learning but now widely used for any parallel
    or distributed Python workload including LLM serving.""",

    """FAISS (Facebook AI Similarity Search) provides highly efficient
    vector similarity search.  It is the backbone of many production RAG
    (Retrieval-Augmented Generation) pipelines.""",

    """Load balancing distributes incoming requests across multiple compute
    nodes to improve throughput, reduce latency, and prevent any single node
    from becoming a bottleneck.""",

    """GPU clusters improve LLM inference throughput by running multiple
    model replicas in parallel.  Techniques like tensor parallelism and
    pipeline parallelism split large models across multiple GPUs.""",

    """Ollama is a lightweight tool for running open-source LLMs locally.
    It supports models like Llama, Phi, Gemma, and Mistral and exposes a
    REST API compatible with most inference frameworks.""",

    """Retrieval-Augmented Generation (RAG) combines a retrieval system with
    a generative model.  The retriever finds relevant context from a
    knowledge base; the generator uses that context to produce a grounded
    answer, reducing hallucinations.""",

    """Heartbeat mechanisms in distributed systems allow a master node to
    detect worker failures.  If a worker stops sending heartbeats within a
    timeout window, the master marks it dead and redirects traffic.""",

    """Circuit breakers in microservices prevent cascading failures by
    temporarily blocking requests to a service that has exceeded a failure
    threshold, giving it time to recover.""",

    """ngrok is a reverse proxy tool that creates secure public tunnels to
    localhost.  It is commonly used in Colab notebooks to expose local
    FastAPI or Flask servers to the public internet.""",

    """FastAPI is a high-performance Python web framework built on top of
    Starlette and Pydantic.  It supports async endpoints out of the box,
    making it ideal for IO-bound workloads like LLM inference proxy.""",

    """Round-robin scheduling assigns each new request to the next worker in
    a circular list, ensuring even distribution when all workers have equal
    capacity and processing time.""",

    """Least-connections scheduling routes each new request to the worker
    with the fewest active connections, adapting dynamically to varying
    request processing times.""",

    """Semaphores are concurrency primitives that limit the number of
    coroutines or threads accessing a shared resource simultaneously.
    asyncio.Semaphore is used in FastAPI to cap concurrent upstream calls.""",

    """AWS EC2 (Elastic Compute Cloud) provides scalable virtual machines.
    A t3.medium instance offers 2 vCPUs and 4 GB RAM, sufficient to host
    a FastAPI master scheduler with NGINX as a reverse proxy.""",
]

print(f"✅ Knowledge base loaded: {len(KNOWLEDGE_BASE)} documents")

✅ Knowledge base loaded: 15 documents


## Cell 6 — Ray Actor: GPUWorker
Each actor owns its own embedding model, FAISS index, and metrics counters.
They share the single Ollama process but send concurrent HTTP requests to it,
which Ollama queues internally.

In [6]:
# ── CELL: GPUWorker v3 — aggressive token cap + Ollama options ────────────────
import ray

@ray.remote(num_gpus=GPU_FRACTION_PER_WORKER)
class GPUWorker:

    def __init__(self, worker_id: str, model_name: str, knowledge_base: list):
        import numpy as np
        import faiss
        import requests
        from sentence_transformers import SentenceTransformer

        self.worker_id   = worker_id
        self.model_name  = model_name
        self.ollama_url  = "http://localhost:11434/api/generate"

        self.active_requests    = 0
        self.completed_requests = 0
        self.failed_requests    = 0
        self.status             = "idle"

        self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.documents       = knowledge_base

        embeddings = self.embedding_model.encode(knowledge_base)
        dim        = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(np.array(embeddings).astype("float32"))

        print(f"[{worker_id}] ✅ ready")
        self._requests = requests
        self._np       = np

    def _retrieve_context(self, query: str, top_k: int = 2) -> str:
        q_emb = self.embedding_model.encode([query])
        _, idxs = self.index.search(
            self._np.array(q_emb).astype("float32"), top_k
        )
        return "\n".join(
            self.documents[i] for i in idxs[0] if i < len(self.documents)
        )

    def _ask_llm(self, prompt: str) -> str:
        resp = self._requests.post(
            self.ollama_url,
            json={
                "model"  : self.model_name,
                "prompt" : prompt,
                "stream" : False,
                "options": {
                    "num_predict" : 80,    # hard cap: ~80 tokens max output
                    "temperature" : 0.0,   # greedy decode = fastest possible
                    "top_k"       : 1,     # greedy
                    "top_p"       : 1.0,
                    "num_ctx"     : 512,   # smaller context window = faster prefill
                },
            },
            timeout=60,
        )
        resp.raise_for_status()
        return resp.json()["response"].strip()

    def _rag_pipeline(self, query: str) -> str:
        context = self._retrieve_context(query)
        prompt  = f"Context: {context}\nQ: {query}\nA:"   # minimal prompt
        return self._ask_llm(prompt)

    def process_query(self, query: str) -> dict:
        import time
        self.active_requests += 1
        self.status = "busy"
        start = time.time()
        try:
            response = self._rag_pipeline(query)
            self.completed_requests += 1
            return {
                "worker_id" : self.worker_id,
                "response"  : response,
                "latency_s" : round(time.time() - start, 3),
                "status"    : "ok",
            }
        except Exception as exc:
            self.failed_requests += 1
            return {
                "worker_id" : self.worker_id,
                "error"     : str(exc),
                "latency_s" : round(time.time() - start, 3),
                "status"    : "error",
            }
        finally:
            self.active_requests = max(0, self.active_requests - 1)
            self.status = "idle" if self.active_requests == 0 else "busy"

    def get_metrics(self) -> dict:
        return {
            "active_requests"    : self.active_requests,
            "completed_requests" : self.completed_requests,
            "failed_requests"    : self.failed_requests,
            "status"             : self.status,
        }

    def ping(self) -> str:
        return f"{self.worker_id}:alive"

print("✅ GPUWorker v3 defined")
print("   num_predict=80, temperature=0.0, num_ctx=512")

✅ GPUWorker v3 defined
   num_predict=80, temperature=0.0, num_ctx=512


## Cell 7 — Spawn N Virtual Worker Actors

In [7]:
# ── CELL 7: Spawn N virtual worker actors ─────────────────────────────────────
import time

ACTORS = {}   # reset — clear any stale handles from previous runs

for i in range(1, NUM_VIRTUAL_WORKERS + 1):
    worker_id = f"{WORKER_BASE_NAME}-{i}"
    print(f"Spawning actor: {worker_id}  (GPU fraction={GPU_FRACTION_PER_WORKER})")
    actor = GPUWorker.remote(worker_id, MODEL_NAME, KNOWLEDGE_BASE)
    ACTORS[worker_id] = actor

# Wait for all actors to finish initialisation
# (SentenceTransformer + FAISS index build takes ~10s per actor)
print("\nWaiting for actors to initialise (loading embedding models)...")
pings = ray.get([a.ping.remote() for a in ACTORS.values()])
for p in pings:
    print(f"  {p}")

print(f"\n✅ {NUM_VIRTUAL_WORKERS} virtual workers ready")
print(f"   Actor IDs: {list(ACTORS.keys())}")

Spawning actor: gpu-worker-1  (GPU fraction=0.3)
Spawning actor: gpu-worker-2  (GPU fraction=0.3)
Spawning actor: gpu-worker-3  (GPU fraction=0.3)

Waiting for actors to initialise (loading embedding models)...
  gpu-worker-1:alive
  gpu-worker-2:alive
  gpu-worker-3:alive

✅ 3 virtual workers ready
   Actor IDs: ['gpu-worker-1', 'gpu-worker-2', 'gpu-worker-3']


## Cell 8 — Create cloudflare Tunnels (one per virtual worker)

In [8]:
import subprocess, threading, time, re

# worker_id → public URL
PUBLIC_URLS = {}
PORTS       = {}
CF_PROCESSES = {}   # keep references so we can kill them on shutdown

def start_cloudflare_tunnel(worker_id: str, port: int) -> str:
    """
    Start a cloudflared quick tunnel for one worker.
    Returns the public https URL.
    Quick tunnels (--url) need no account or login —
    they give you a random *.trycloudflare.com URL that
    never rate-limits and never expires during the session.
    """
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}",
         "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    CF_PROCESSES[worker_id] = proc

    # Parse the public URL from cloudflared output
    url = None
    for line in proc.stdout:
        # cloudflared prints the URL in a line like:
        # "Your quick Tunnel has been created! Visit it at: https://..."
        match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            break
        # Timeout safety — if no URL found in 30 lines something is wrong
    return url


print(f"Starting {NUM_VIRTUAL_WORKERS} Cloudflare tunnels...\n")

for idx, worker_id in enumerate(ACTORS.keys(), start=1):
    port = BASE_PORT + idx - 1
    PORTS[worker_id] = port

    print(f"  {worker_id} (port {port})... ", end="", flush=True)
    url = start_cloudflare_tunnel(worker_id, port)

    if url:
        PUBLIC_URLS[worker_id] = url
        print(f"✅ {url}")
    else:
        print(f"❌ Failed to get URL")

print(f"\n✅ {len(PUBLIC_URLS)} Cloudflare tunnels active")

Starting 3 Cloudflare tunnels...

  gpu-worker-1 (port 8001)... ✅ https://guitar-representatives-angel-family.trycloudflare.com
  gpu-worker-2 (port 8002)... ✅ https://matched-mirror-advanced-walks.trycloudflare.com
  gpu-worker-3 (port 8003)... ✅ https://pray-blessed-wedding-suppose.trycloudflare.com

✅ 3 Cloudflare tunnels active


## Cell 9 — FastAPI Apps (one per virtual worker)
Each app forwards `/process` calls to its dedicated Ray actor.

In [9]:
# ── CELL: FastAPI servers — queue requests, no hard 503 rejection ─────────────
import uvicorn, asyncio, threading, nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
import ray, time

nest_asyncio.apply()

# Queue depth per worker — requests wait instead of getting 503
# With phi3:mini ~4s per request, depth=6 means max ~24s queue wait
QUEUE_DEPTH = 6

class QueryRequest(BaseModel):
    query: str


def make_app(worker_id: str, actor) -> FastAPI:
    app = FastAPI(title=f"Worker {worker_id}")
    sem = asyncio.Semaphore(QUEUE_DEPTH)

    @app.get("/")
    async def health():
        return {"status": "alive", "worker_id": worker_id}

    @app.get("/metrics")
    async def get_metrics():
        loop = asyncio.get_event_loop()
        metrics = await loop.run_in_executor(
            None, lambda: ray.get(actor.get_metrics.remote())
        )
        return metrics

    @app.get("/queue_depth")
    async def queue_depth():
        return {
            "worker_id"   : worker_id,
            "free_slots"  : sem._value,
            "queue_depth" : QUEUE_DEPTH,
        }

    @app.post("/process")
    async def process(req: QueryRequest):
        # Wait for a slot — never reject, just queue
        async with sem:
            loop = asyncio.get_event_loop()
            result = await loop.run_in_executor(
                None, lambda: ray.get(actor.process_query.remote(req.query))
            )
        return result

    return app


# Kill old servers on those ports
import subprocess
for port in PORTS.values():
    subprocess.run(["fuser", "-k", f"{port}/tcp"],
                   capture_output=True)
time.sleep(1)

SERVER_THREADS = {}
for worker_id, actor in ACTORS.items():
    port = PORTS[worker_id]
    app  = make_app(worker_id, actor)

    def _run(a=app, p=port):
        uvicorn.run(a, host="0.0.0.0", port=p, log_level="warning")

    t = threading.Thread(target=_run, daemon=True, name=f"srv-{worker_id}")
    t.start()
    SERVER_THREADS[worker_id] = t
    print(f"  {worker_id} started on port {port}  (queue_depth={QUEUE_DEPTH})")

time.sleep(3)

# Health check + queue status
import requests as req
print("\nHealth + queue checks:")
for worker_id in ACTORS:
    port = PORTS[worker_id]
    try:
        h = req.get(f"http://localhost:{port}/", timeout=5).json()
        q = req.get(f"http://localhost:{port}/queue_depth", timeout=5).json()
        print(f"  {worker_id}  status={h['status']}  "
              f"free_slots={q['free_slots']}/{q['queue_depth']}")
    except Exception as e:
        print(f"  {worker_id}  ERROR: {e}")

print("\n✅ All FastAPI servers running — requests queue instead of rejecting")

  gpu-worker-1 started on port 8001  (queue_depth=6)
  gpu-worker-2 started on port 8002  (queue_depth=6)
  gpu-worker-3 started on port 8003  (queue_depth=6)

Health + queue checks:
  gpu-worker-1  status=alive  free_slots=6/6
  gpu-worker-2  status=alive  free_slots=6/6
  gpu-worker-3  status=alive  free_slots=6/6

✅ All FastAPI servers running — requests queue instead of rejecting


## Cell 10 — Register All Virtual Workers with the Master

In [10]:
import requests as req

print(f"Registering {NUM_VIRTUAL_WORKERS} workers with master at {MASTER_API_URL}\n")

for worker_id, public_url in PUBLIC_URLS.items():
    payload = {
        "worker_name" : worker_id,
        "worker_url"  : public_url,
        "gpu"         : "colab-T4-virtual",
        "model"       : MODEL_NAME,
        "status"      : "idle",
    }
    try:
        r = req.post(f"{MASTER_API_URL}/register_worker", json=payload, timeout=10)
        print(f"  {worker_id:20s}  {r.status_code}  {r.json()}")
    except Exception as e:
        print(f"  {worker_id:20s}  ERROR: {e}")

print("\n✅ Registration complete")

# Verify via /workers endpoint
try:
    workers_resp = req.get(f"{MASTER_API_URL}/workers", timeout=10)
    registered   = list(workers_resp.json().keys())
    print(f"\nMaster now knows: {registered}")
except Exception as e:
    print(f"Could not verify: {e}")

Registering 3 workers with master at http://13.63.238.244

  gpu-worker-1          201  {'message': 'worker registered', 'worker_name': 'gpu-worker-1'}
  gpu-worker-2          201  {'message': 'worker re-registered', 'worker_name': 'gpu-worker-2'}
  gpu-worker-3          201  {'message': 'worker re-registered', 'worker_name': 'gpu-worker-3'}

✅ Registration complete

Master now knows: ['abdooo1-worker-2', 'abdooo1-worker-1', 'abdo-worker-2', 'gpu-worker-2', 'gpu-worker-3', 'lllllllll-worker-2', 'lllllllll-worker-3', 'lllllllll-worker-1', 'vvvddd-worker-2', 'vvvddd-worker-3', 'gpu-worker-1']


## Cell 11 — Heartbeat Threads (one per virtual worker)
Each worker sends its own metrics to the master every `HEARTBEAT_SECONDS` seconds.
This keeps the master's failure detector happy and updates the load-aware scheduler.

In [11]:
import threading
import time
import requests as req

_heartbeat_active = True   # set to False to stop all heartbeats


def heartbeat_loop(worker_id: str, actor):
    """Continuously send heartbeats for one worker to the master."""
    print(f"[{worker_id}] Heartbeat thread started")
    while _heartbeat_active:
        try:
            # Get metrics from the Ray actor
            metrics = ray.get(actor.get_metrics.remote())

            payload = {
                "worker_name" : worker_id,
                "metrics"     : metrics,
            }
            r = req.post(
                f"{MASTER_API_URL}/heartbeat",
                json=payload,
                timeout=8
            )
            if r.status_code != 200:
                print(f"[{worker_id}] Heartbeat warn: {r.text}")

        except Exception as e:
            print(f"[{worker_id}] Heartbeat error: {e}")

        time.sleep(HEARTBEAT_SECONDS)


HEARTBEAT_THREADS = {}

for worker_id, actor in ACTORS.items():
    t = threading.Thread(
        target=heartbeat_loop,
        args=(worker_id, actor),
        daemon=True,
        name=f"hb-{worker_id}",
    )
    t.start()
    HEARTBEAT_THREADS[worker_id] = t

print(f"✅ {NUM_VIRTUAL_WORKERS} heartbeat threads started")
print(f"   Each pings master every {HEARTBEAT_SECONDS}s")

[gpu-worker-1] Heartbeat thread started
[gpu-worker-2] Heartbeat thread started
[gpu-worker-3] Heartbeat thread started
✅ 3 heartbeat threads started
   Each pings master every 8s


## Cell 12 — System Status Dashboard
Run this cell any time to see a live snapshot of all workers.

In [12]:
import requests as req
import ray

print("=" * 70)
print(" SYSTEM STATUS DASHBOARD")
print("=" * 70)

# ── Local actor metrics ──────────────────────────────────────────────────────
print("\n📊 LOCAL ACTOR METRICS (from Ray):")
print(f"  {'Worker':22s} {'Active':>8} {'Done':>8} {'Failed':>8} {'Status':>10}")
print("  " + "-" * 60)

all_metrics = ray.get([a.get_metrics.remote() for a in ACTORS.values()])
for worker_id, m in zip(ACTORS.keys(), all_metrics):
    print(f"  {worker_id:22s} {m['active_requests']:>8} {m['completed_requests']:>8} "
          f"{m['failed_requests']:>8} {m['status']:>10}")

# ── Master view ──────────────────────────────────────────────────────────────
print("\n🖥️  MASTER VIEW (from AWS EC2):")
try:
    workers_resp  = req.get(f"{MASTER_API_URL}/workers",  timeout=8).json()
    metrics_resp  = req.get(f"{MASTER_API_URL}/metrics",  timeout=8).json()

    print(f"  Total workers  : {metrics_resp['workers_total']}")
    print(f"  Alive          : {metrics_resp['workers_alive']}")
    print(f"  Dead           : {metrics_resp['workers_dead']}")
    print(f"  Requests recv  : {metrics_resp['requests_received']}")
    print(f"  Total done     : {metrics_resp['total_completed']}")
    print(f"  Scheduler      : {metrics_resp['scheduler']}")

    print(f"\n  {'Worker':22s} {'Alive':>6} {'Active':>8} {'Done':>8}")
    print("  " + "-" * 50)
    for wname, wdata in workers_resp.items():
        print(f"  {wname:22s} {str(wdata['alive']):>6} "
              f"{wdata['active_requests']:>8} {wdata['completed_requests']:>8}")
except Exception as e:
    print(f"  Could not reach master: {e}")

# ── Public URLs ──────────────────────────────────────────────────────────────
print("\n🌐 PUBLIC URLs (ngrok):")
for wid, url in PUBLIC_URLS.items():
    print(f"  {wid:22s}  {url}")

print("\n" + "=" * 70)

 SYSTEM STATUS DASHBOARD

📊 LOCAL ACTOR METRICS (from Ray):
  Worker                   Active     Done   Failed     Status
  ------------------------------------------------------------
  gpu-worker-1                  0        0        0       idle
  gpu-worker-2                  0        0        0       idle
  gpu-worker-3                  0        0        0       idle

🖥️  MASTER VIEW (from AWS EC2):
  Total workers  : 11
  Alive          : 5
  Dead           : 6
  Requests recv  : 8487
  Could not reach master: 'total_completed'

🌐 PUBLIC URLs (ngrok):
  gpu-worker-1            https://guitar-representatives-angel-family.trycloudflare.com
  gpu-worker-2            https://matched-mirror-advanced-walks.trycloudflare.com
  gpu-worker-3            https://pray-blessed-wedding-suppose.trycloudflare.com



## Cell 13 — Local Query Test
Test a single query directly against a local actor (bypassing the master).

In [13]:
import ray, time

TEST_QUERY = "What is load balancing and how does it improve distributed systems?"

# Pick the first actor
first_worker_id = list(ACTORS.keys())[0]
actor = ACTORS[first_worker_id]

print(f"Sending test query to: {first_worker_id}")
print(f"Query: {TEST_QUERY}\n")

start  = time.time()
result = ray.get(actor.process_query.remote(TEST_QUERY))
elapsed = time.time() - start

print(f"Worker ID : {result['worker_id']}")
print(f"Latency   : {result.get('latency_s', elapsed):.2f}s")
print(f"Status    : {result.get('status', '?')}")
print(f"\n--- Response ---")
print(result.get('response', result.get('error', 'N/A')))

Sending test query to: gpu-worker-1
Query: What is load balancing and how does it improve distributed systems?

Worker ID : gpu-worker-1
Latency   : 3.36s
Status    : ok

--- Response ---
Load balancing distributes incoming requests across multiple compute nodes to improve throughput, reduce latency, and prevent any single node from becoming a bottleneck in distributed systems. It allows for efficient resource allocation and fault tolerance by distributing the workload over multiple nodes, ensuring that each node can handle a portion of the load without becoming overwhelmed or lagging behind. This helps in achieving scalability, fault tolerance, and high availability, which are


## Cell 14 — End-to-End Test via Master
Send a query through the full distributed pipeline:
Client → Master (EC2) → Load Balancer → This Colab Worker → Ollama LLM

In [14]:
import requests as req
import time

TEST_QUERY = "Explain how Ray helps with GPU cluster task distribution."

print(f"Sending query via master at {MASTER_API_URL}")
print(f"Query: {TEST_QUERY}\n")

start = time.time()
try:
    r = req.post(
        f"{MASTER_API_URL}/submit_task",
        json={"query": TEST_QUERY},
        timeout=180
    )
    elapsed = time.time() - start
    data = r.json()

    print(f"Request ID      : {data.get('request_id')}")
    print(f"Scheduler       : {data.get('scheduler')}")
    print(f"Selected worker : {data.get('selected_worker')}")
    print(f"Attempts        : {data.get('attempts')}")
    print(f"Round-trip time : {elapsed:.2f}s")
    if 'result' in data:
        inner = data['result']
        print(f"Worker latency  : {inner.get('latency_s')}s")
        print(f"\n--- LLM Response ---")
        print(inner.get('response', inner.get('error', 'N/A')))
    elif 'error' in data:
        print(f"\n❌ Error: {data['error']}")
        print(f"   Tried: {data.get('tried_workers')}")
except Exception as e:
    print(f"Request failed: {e}")

Sending query via master at http://13.63.238.244
Query: Explain how Ray helps with GPU cluster task distribution.

Request ID      : 8488
Scheduler       : load_aware
Selected worker : gpu-worker-2
Attempts        : 1
Round-trip time : 1.36s
Worker latency  : 0.7s

--- LLM Response ---
Ray provides distributed execution frameworks that allow for efficient computation on large-scale, multi-core GPUs. By leveraging Ray's capabilities, GPUs can be used to distribute tasks across multiple cores and compute resources, thereby improving the throughput of LLM serving tasks. This allows for faster processing times and increased efficiency in handling large volumes of data.


In [15]:
# ── CELL: Warmup — pre-heat Ollama model cache ────────────────────────────────
# Sending a cold request to a fresh Ollama load can add 2-5s of model-load
# overhead on the first call.  Fire one warmup query per worker first.

import ray, time, requests as req

WARMUP_QUERY = "What is distributed computing? Answer in one sentence."

print("🔥 Warming up workers...")
warmup_start = time.time()

futures = {wid: actor.process_query.remote(WARMUP_QUERY)
           for wid, actor in ACTORS.items()}

for wid, fut in futures.items():
    result = ray.get(fut)
    print(f"  {wid:22s}  latency={result.get('latency_s')}s  "
          f"status={result.get('status')}")

print(f"\n✅ Warmup complete in {time.time()-warmup_start:.1f}s — workers are hot")

🔥 Warming up workers...
  gpu-worker-1            latency=1.187s  status=ok
  gpu-worker-2            latency=1.344s  status=ok
  gpu-worker-3            latency=1.538s  status=ok

✅ Warmup complete in 1.5s — workers are hot


## Cell 15 — Concurrent Load Test
Simulate multiple users hitting the master simultaneously.
Watch the master distribute them across your virtual workers.

In [18]:
import threading
import time
import requests as req
from collections import defaultdict

# ── Load test parameters ─────────────────────────────────────────────────────
NUM_REQUESTS    = 120          # total requests to fire
CONCURRENCY     = 4           # simultaneous threads
REQUEST_TIMEOUT = 180         # seconds per request

QUERIES = [
    "What is FAISS and how is it used in RAG?",
    "How does Round Robin scheduling work?",
    "Explain the circuit breaker pattern.",
    "What is a heartbeat in distributed systems?",
    "How does GPU parallelism speed up LLM inference?",
    "What is Ollama?",
    "Explain load-aware scheduling vs least connections.",
    "What is a semaphore in async programming?",
    "How does RAG reduce hallucinations in LLMs?",
    "What is ngrok used for?",
    "Describe fault tolerance in distributed systems.",
    "What is AWS EC2?",
]

# ── Results storage ──────────────────────────────────────────────────────────
results_lock  = threading.Lock()
results       = []            # list of dicts
worker_counts = defaultdict(int)


def send_request(req_id: int, query: str):
    start = time.time()
    try:
        r = req.post(
            f"{MASTER_API_URL}/submit_task",
            json={"query": query},
            timeout=REQUEST_TIMEOUT
        )
        data    = r.json()
        elapsed = time.time() - start
        worker  = data.get("selected_worker", "unknown")

        entry = {
            "id"      : req_id,
            "worker"  : worker,
            "latency" : round(elapsed, 2),
            "ok"      : "error" not in data,
        }
        with results_lock:
            results.append(entry)
            worker_counts[worker] += 1
            print(f"  [{req_id:03d}] worker={worker:22s}  latency={elapsed:.2f}s  ok={entry['ok']}")

    except Exception as e:
        elapsed = time.time() - start
        with results_lock:
            results.append({"id": req_id, "worker": "ERR", "latency": elapsed, "ok": False})
            print(f"  [{req_id:03d}] ERROR: {e}")


# ── Run with thread pool ─────────────────────────────────────────────────────
print(f"Starting load test: {NUM_REQUESTS} requests, concurrency={CONCURRENCY}")
print("-" * 70)

from concurrent.futures import ThreadPoolExecutor

test_start = time.time()
with ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
    futures = [
        pool.submit(send_request, i, QUERIES[i % len(QUERIES)])
        for i in range(NUM_REQUESTS)
    ]
    for f in futures:
        f.result()   # propagate exceptions
test_elapsed = time.time() - test_start

# ── Summary ──────────────────────────────────────────────────────────────────
success = sum(1 for r in results if r["ok"])
latencies = [r["latency"] for r in results if r["ok"]]

print("\n" + "=" * 70)
print(" LOAD TEST SUMMARY")
print("=" * 70)
print(f"  Total requests  : {NUM_REQUESTS}")
print(f"  Successful      : {success}")
print(f"  Failed          : {NUM_REQUESTS - success}")
print(f"  Wall time       : {test_elapsed:.1f}s")
print(f"  Throughput      : {NUM_REQUESTS / test_elapsed:.2f} req/s")
if latencies:
    print(f"  Avg latency     : {sum(latencies)/len(latencies):.2f}s")
    print(f"  Min latency     : {min(latencies):.2f}s")
    print(f"  Max latency     : {max(latencies):.2f}s")
print(f"\n  Worker distribution:")
for wid, cnt in sorted(worker_counts.items()):
    bar = "█" * cnt
    print(f"    {wid:22s}  {bar} ({cnt})")

Starting load test: 120 requests, concurrency=4
----------------------------------------------------------------------
  [000] worker=vvvddd-worker-2         latency=1.59s  ok=True
  [003] worker=vvvddd-worker-3         latency=1.96s  ok=True
  [001] worker=gpu-worker-2            latency=2.19s  ok=True
  [002] worker=gpu-worker-3            latency=2.79s  ok=True
  [004] worker=vvvddd-worker-2         latency=1.37s  ok=True
  [005] worker=gpu-worker-2            latency=1.21s  ok=True
  [006] worker=vvvddd-worker-3         latency=1.78s  ok=True
  [007] worker=gpu-worker-3            latency=1.28s  ok=True
  [008] worker=gpu-worker-2            latency=1.59s  ok=True
  [009] worker=vvvddd-worker-2         latency=1.45s  ok=True
  [010] worker=gpu-worker-3            latency=1.92s  ok=True
  [011] worker=vvvddd-worker-3         latency=2.01s  ok=True
  [012] worker=gpu-worker-2            latency=1.66s  ok=True
  [013] worker=vvvddd-worker-2         latency=1.99s  ok=True
  [016] worke

## Cell 16 — Fault Tolerance Demo
Kill one worker and show the master automatically redirects traffic to survivors.

In [19]:
import ray, requests as req, time

# ── Step 1: Pick the worker to kill ─────────────────────────────────────────
victim_id = list(ACTORS.keys())[0]   # kill first worker
print(f"🔪 Simulating failure of: {victim_id}")

# Kill the Ray actor — its heartbeats will stop
ray.kill(ACTORS[victim_id])
print(f"   Actor killed. Master will detect failure in ~{15}s (HEARTBEAT_TIMEOUT).")
print(f"   Our heartbeat thread will start erroring for {victim_id}.")

# ── Step 2: Immediately send a request — should succeed via other workers ────
print("\n⚡ Sending query immediately after kill (master retries automatically)...")
time.sleep(2)

r = req.post(
    f"{MASTER_API_URL}/submit_task",
    json={"query": "How does fault tolerance work in distributed systems?"},
    timeout=180
)
data = r.json()
print(f"   Selected worker : {data.get('selected_worker')}")
print(f"   Attempts        : {data.get('attempts')}")
print(f"   Success         : {'error' not in data}")

# ── Step 3: Wait for master failure detector ─────────────────────────────────
print(f"\n⏳ Waiting 20s for master failure detector to mark {victim_id} dead...")
time.sleep(20)

workers_data = req.get(f"{MASTER_API_URL}/workers", timeout=8).json()
print(f"\n   Master worker states:")
for wname, winfo in workers_data.items():
    alive = "✅ alive" if winfo["alive"] else "❌ dead"
    print(f"   {wname:22s}  {alive}")

# ── Step 4: Deregister victim so it doesn't count against the master ─────────
req.delete(f"{MASTER_API_URL}/workers/{victim_id}", timeout=8)
print(f"\n   {victim_id} deregistered from master")
del ACTORS[victim_id]

print("\n✅ Fault tolerance demo complete. Remaining workers continue serving.")

🔪 Simulating failure of: gpu-worker-1
   Actor killed. Master will detect failure in ~15s (HEARTBEAT_TIMEOUT).
   Our heartbeat thread will start erroring for gpu-worker-1.

⚡ Sending query immediately after kill (master retries automatically)...
   Selected worker : gpu-worker-2
   Attempts        : 1
   Success         : True

⏳ Waiting 20s for master failure detector to mark gpu-worker-1 dead...
[gpu-worker-1] Heartbeat error: The actor died unexpectedly before finishing this task.
	class_name: GPUWorker
	actor_id: 2510d0bd94b1bc37786c135e01000000
	pid: 15976
	namespace: 1c33f4a6-d635-405f-a233-f88a79b7c3fc
	ip: 172.28.0.12
The actor is dead because it was killed by `ray.kill`.
[gpu-worker-1] Heartbeat error: The actor died unexpectedly before finishing this task.
	class_name: GPUWorker
	actor_id: 2510d0bd94b1bc37786c135e01000000
	pid: 15976
	namespace: 1c33f4a6-d635-405f-a233-f88a79b7c3fc
	ip: 172.28.0.12
The actor is dead because it was killed by `ray.kill`.
[gpu-worker-1] Heartbe

## Cell 17 — Switch Scheduling Algorithm
Change the master's scheduler at runtime.

In [20]:
import requests as req

# Options: round_robin | least_connections | load_aware
NEW_SCHEDULER = "load_aware"

r = req.post(f"{MASTER_API_URL}/scheduler/{NEW_SCHEDULER}", timeout=8)
print(r.json())

# Verify
health = req.get(f"{MASTER_API_URL}/", timeout=8).json()
print(f"Master scheduler is now: {health['scheduler']}")

{'message': 'scheduler changed to load_aware'}
Master scheduler is now: load_aware


## Cell 18 — Graceful Shutdown
Run this before ending the Colab session to cleanly deregister all workers.

In [ ]:
import requests as req

# Stop heartbeat threads
_heartbeat_active = False
print("Heartbeat threads stopping...")

# Deregister all still-alive workers
for worker_id in list(ACTORS.keys()):
    try:
        r = req.delete(f"{MASTER_API_URL}/workers/{worker_id}", timeout=8)
        print(f"  Deregistered {worker_id}: {r.json()}")
    except Exception as e:
        print(f"  {worker_id}: {e}")

# Kill all Ray actors
for actor in ACTORS.values():
    try:
        ray.kill(actor)
    except Exception:
        pass

# Close ngrok tunnels
from pyngrok import ngrok
ngrok.kill()

# Shutdown Ray
ray.shutdown()

print("\n✅ Graceful shutdown complete")